# Hugging Face NLP Tasks 

## pipeline

`pipeline`은 Hugging Face에서 가장 간단하게 모델을 테스트할 수 있는 인터페이스이다.

`pipeline`은 내부적으로 다음 과정을 처리한다.

1. task에 맞는 모델 클래스 선택
2. tokenizer 로드
3. model 로드
4. 입력 문장 전처리
5. 모델 추론
6. 출력 후처리

`pipeline`은 사전학습 모델을 빠르게 테스트해보는 편의 기능에 가깝다.
또한 모든 task가 항상 pipeline으로 제공되는 것은 아니다.

In [1]:
from transformers import pipeline
from transformers.pipelines import SUPPORTED_TASKS

sorted(SUPPORTED_TASKS.keys())

['any-to-any',
 'audio-classification',
 'automatic-speech-recognition',
 'depth-estimation',
 'document-question-answering',
 'feature-extraction',
 'fill-mask',
 'image-classification',
 'image-feature-extraction',
 'image-segmentation',
 'image-text-to-text',
 'keypoint-matching',
 'mask-generation',
 'object-detection',
 'table-question-answering',
 'text-classification',
 'text-generation',
 'text-to-audio',
 'token-classification',
 'video-classification',
 'zero-shot-audio-classification',
 'zero-shot-classification',
 'zero-shot-image-classification',
 'zero-shot-object-detection']

## Fill-Mask

BERT는 문장을 왼쪽에서 오른쪽으로 생성하는 모델이 아니다. 
BERT는 문장 중간의 일부 토큰을 `[MASK]`로 가린 뒤, 주변 문맥을 보고 해당 위치에 들어갈 단어를 예측하도록 사전학습되었다.
이 작업을 Masked Language Modeling, 줄여서 MLM이라고 한다.

In [2]:
fill_mask = pipeline(task='fill-mask', model='klue/bert-base')

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
fill_mask("대한민국의 수도는 [MASK]이다.")

[{'score': 0.5954610705375671,
  'token': 3671,
  'token_str': '서울',
  'sequence': '대한민국의 수도는 서울 이다.'},
 {'score': 0.0810634046792984,
  'token': 9474,
  'token_str': '광화문',
  'sequence': '대한민국의 수도는 광화문 이다.'},
 {'score': 0.04720195755362511,
  'token': 7141,
  'token_str': '평양',
  'sequence': '대한민국의 수도는 평양 이다.'},
 {'score': 0.029445303604006767,
  'token': 3902,
  'token_str': '부산',
  'sequence': '대한민국의 수도는 부산 이다.'},
 {'score': 0.02646547555923462,
  'token': 4068,
  'token_str': '인천',
  'sequence': '대한민국의 수도는 인천 이다.'}]

## Text Classification

문장 분류는 입력 문장 전체에 하나의 라벨을 붙이는 작업이다.
예를 들어 다음과 같은 작업이 여기에 해당한다.

1. 감성 분석
2. 스팸 분류
3. 뉴스 주제 분류
4. 리뷰 긍정/부정 분류

앞서 `klue/bert-base`에 분류 헤드를 붙여 NSMC 감성분석 모델을 직접 fine-tuning했다. 
여기서는 기존 공개 모델을 사용하여 pipeline의 동작 흐름만 확인한다.

In [4]:
# 영어 감성분석용으로 fine tuning 된 공개 모델 사용
classifier = pipeline(task='text-classification', model='distilbert/distilbert-base-uncased-finetuned-sst-2-english')

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [5]:
texts = [
    'This movie was fantastic.',
    'The story was boring and too long.'
]

classifier(texts)

[{'label': 'POSITIVE', 'score': 0.9998798370361328},
 {'label': 'NEGATIVE', 'score': 0.9997686743736267}]

## Token Classification

토큰 분류는 문장 전체에 하나의 라벨을 붙이는 것이 아니라, 각 토큰마다 라벨을 붙이는 작업이다.

대표적인 예시는 NER(Named Entity Recognition, 개체명 인식)이다.

NER은 문장에서 사람, 장소, 기관명 같은 개체를 찾아내는 작업이다.

예:

`Barack Obama was born in Hawaii.`

- Barack Obama → 사람
- Hawaii → 장소

In [6]:
# aggregation_strategy='simple' 속성은 subword 단위 결과를 단어 단위로 묶어준다.
ner = pipeline(task='token-classification', model='dslim/bert-base-NER', aggregation_strategy='simple')

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--dslim--bert-base-NER. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
sentence = 'Barack Obama was born in Hawaii and worked in Washington.'

ner(sentence)

[{'entity_group': 'PER',
  'score': np.float32(0.9993496),
  'word': 'Barack Obama',
  'start': 0,
  'end': 12},
 {'entity_group': 'LOC',
  'score': np.float32(0.9997689),
  'word': 'Hawaii',
  'start': 25,
  'end': 31},
 {'entity_group': 'LOC',
  'score': np.float32(0.9996375),
  'word': 'Washington',
  'start': 46,
  'end': 56}]

## Feature Extraction

특징 추출은 모델의 마지막 분류 결과가 아니라, 중간 은닉 상태를 벡터로 얻는 작업이다.

이 벡터는 다음과 같은 작업에 활용할 수 있다.

1. 문장 유사도 계산
2. 검색 시스템의 임베딩
3. 클러스터링
4. 다른 모델의 입력 feature

다만 실제 서비스 수준의 문장 임베딩은 `sentence-transformers` 계열 모델을 사용하는 경우가 많다. 

In [ ]:
import torch 
from transformers import AutoTokenizer, AutoModel 

model_name = 'bert-base-multilingual-cased'

tokenizer = AutoTokenizer.from_pretrained(model_name)
# AutoModel은 특정 task head 없이 기본 transformer 본체만 불러온다. 
# 분류 헤드나 MLM 헤드 없이 은닉 상태를 얻는 용도로 사용할 수 있다.
model = AutoModel.from_pretrained(model_name)

model.eval()

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
text = '자연어 처리는 인간의 언어를 컴퓨터가 처리하도록 만드는 기술이다.'

inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs)

# (batch_size, seq_len, hidden_size)
last_hidden_state = outputs.last_hidden_state
last_hidden_state.shape

torch.Size([1, 25, 768])

In [10]:
# 필요한 경우 설치
%pip install -U sentence-transformers -q

Note: you may need to restart the kernel to use updated packages.


In [11]:
# 실제 문장 임베딩의 품질을 높이려면 sentence-transformers를 사용하는 것이 일반적이다.
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 문장 임베딩에 특화 된 공개 모델 사용
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

sentences = [
    '오늘 영화는 정말 재미있었다.',
    '이 영화는 매우 흥미로웠다.',
    '오늘 점심으로 김치찌개를 먹었다.'
]

# 각 문장을 하나의 벡터로 변환한다.
embeddings = model.encode(sentences)

embeddings.shape

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(3, 384)

In [12]:
import pandas as pd

# 문장 간 코사인 유사도 계산
similarity_matrix = cosine_similarity(embeddings)

pd.DataFrame(similarity_matrix, index=sentences, columns=sentences)

,오늘 영화는 정말 재미있었다.,이 영화는 매우 흥미로웠다.,오늘 점심으로 김치찌개를 먹었다.
오늘 영화는 정말 재미있었다.,1.000000,0.713746,0.289016
이 영화는 매우 흥미로웠다.,0.713746,1.000000,0.045346
오늘 점심으로 김치찌개를 먹었다.,0.289016,0.045346,1.000000


## Translation: Seq2Seq 모델로 번역하기

번역은 대표적인 Seq2Seq 문제이다.

입력: 한국어 문장  
출력: 영어 문장

여기서는 한국어를 영어로 번역하는 공개 모델인 `Helsinki-NLP/opus-mt-ko-en`을 사용한다.

In [13]:
from transformers import AutoModelForSeq2SeqLM

model_name = 'Helsinki-NLP/opus-mt-ko-en'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.eval()

config.json: 0.00B [00:00, ?B/s]

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-ko-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(65001, 512, padding_idx=65000)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(65001, 512, padding_idx=65000)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [14]:
text = '오늘은 자연어 딥러닝 수업의 마지막 시간입니다.'

inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)

with torch.no_grad():
    # generate : decoder가 출력 토큰을 순차적으로 생성하도록 한다.
    # max_new_tokens : 새로 생성할 최대 토큰 수
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50
    )

# 특수 토큰은 결과에서 제거하고 문자열로 변환한다.
translated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
translated_text

Both `max_new_tokens` (=50) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Today is the last time in the natural language deepning class.'

### generate 주요 인자

`generate()`에는 생성 방식을 조절하는 여러 인자가 있다.

1. `max_new_tokens`: 새로 생성할 최대 토큰 수
2. `num_beams`: Beam Search에서 유지할 후보 문장 개수
3. `do_sample`: 확률 기반 샘플링 사용 여부
4. `temperature`: 샘플링 시 확률 분포를 얼마나 날카롭게 또는 부드럽게 볼지 조절
5. `top_k`: 확률이 높은 상위 k개 후보 중에서 샘플링
6. `top_p`: 누적 확률 p 안에 들어오는 후보들 중에서 샘플링
7. `early_stopping`: 종료 조건을 만족하면 생성을 일찍 끝낼지 여부

번역이나 요약처럼 비교적 정답 형태가 정해진 작업에서는 `num_beams`를 사용하는 Beam Search가 자주 사용된다.  
반대로 GPT 기반 글쓰기처럼 다양한 결과가 필요한 작업에서는 `do_sample=True`, `top_k`, `top_p`, `temperature` 같은 옵션을 사용하는 경우가 많다.

In [15]:
# beam search를 사용한 번역 예시
# num_beams가 커지면 여러 후보를 비교하며 생성하므로 결과가 더 안정적일 수 있지만, 속도는 느려질 수 있음
# num_beams=1 은 가장 가능성이 높은 후보만 따라가는 greedy search에 가까움

texts = [
    '밥은 먹었니?',
    '항상 감사하게 생각하고 있습니다.',
    '이 모델은 한국어 문장을 영어로 번역합니다.'
]

inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        num_beams=4,
        early_stopping=True
    )

translated_texts = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
translated_texts

Both `max_new_tokens` (=50) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['Did you eat?',
 "I'm always thankful.",
 'This model translates Korean sentences into English.']

## Summarization: Seq2Seq 모델로 요약하기

요약도 번역과 마찬가지로 입력 텍스트를 바탕으로 새로운 출력 문장을 생성하는 문제이다.

따라서 번역과 동일하게 `AutoModelForSeq2SeqLM`과 `generate()`를 사용할 수 있다.

여기서는 비교적 가벼운 `t5-small` 모델을 사용한다. 
T5는 모든 NLP 문제를 text-to-text 형식으로 바꾸어 처리하는 모델이다.
`t5-small`은 가벼운 실습용 모델이므로 요약 품질이 항상 좋지는 않다.

T5를 사용할 때는 입력 앞에 `summarize:` 같은 task prefix를 붙이는 것이 일반적이다.

In [16]:
model_name = 't5-small'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.eval()

config.json: 0.00B [00:00, ?B/s]

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [17]:
text = """
Natural language processing is a field of artificial intelligence that focuses on the interaction between computers and human language.
It allows computers to process, analyze, and generate text or speech in ways that are useful for people.
Common applications include search engines, machine translation, chatbots, sentiment analysis, spam filtering, and document summarization.

In the past, many natural language processing systems relied on rule-based methods or traditional machine learning algorithms.
These systems often required manually designed features, such as word counts, dictionaries, or grammatical patterns.
Although they worked for some tasks, they were limited when language became ambiguous or context-dependent.

Recent transformer-based models have greatly improved the performance of natural language processing systems.
Models such as BERT, T5, and GPT can learn rich contextual representations from large amounts of text.
As a result, modern NLP systems can perform tasks such as question answering, summarization, translation, and text classification with much higher accuracy than earlier approaches.

However, these models also have limitations.
They require large amounts of data and computing resources, and their outputs can sometimes be biased, incorrect, or difficult to explain.
For this reason, it is important to understand not only how to use NLP models, but also how they work and where their limitations are.
"""

In [18]:
# T5는 task prefix를 통해 어떤 작업을 수행할지 알려주는 방식으로 학습 되었다.
input_text = 'summarize:' + text

inputs = tokenizer(input_text, return_tensors='pt', max_length=512, truncation=True)

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        num_beams=4,
        early_stopping=True
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
summary    

'natural language processing is a field of artificial intelligence that focuses on the interaction between computers and human language. it allows computers to process, analyze, and generate text or speech in ways that are useful for people. common applications include machine translation, chatbots, sentiment analysis, spam filtering.'

## Text Generation: GPT 계열 모델로 이어쓰기

GPT 계열 모델은 Decoder-only 구조이다.

BERT가 문장 전체를 보고 이해하는 데 강하다면, GPT 계열은 앞 문맥을 보고 다음 토큰을 이어 생성하는 데 강하다.

여기서는 `gpt2` 모델을 사용하여 영어 문장 생성을 확인한다.

`gpt2`는 최신 대화형 LLM이 아니라 작은 공개 언어 모델이다.
따라서 결과 품질보다는 Decoder-only 모델이 앞 문맥을 바탕으로 다음 토큰을 이어 생성한다는 흐름을 확인하는 데 초점을 둔다.

In [19]:
from transformers import AutoModelForCausalLM

model_name = 'gpt2'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# gpt2 모델 토크나이저에 기본 pad_token이 없어서 eos_token을 pad_token 처럼 사용한다.
tokenizer.pad_token = tokenizer.eos_token
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [21]:
prompt = 'The most important idea in natural language processing is'

inputs = tokenizer(prompt, return_tensors='pt', padding=True)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
generated_text

'The most important idea in natural language processing is that the words are not just words, but also expressions.\n\nThe first thing to understand about natural language processing is that it is a very simple process.\n\nThe word "I" is a word that is not a word. It is'

## task별 모델 클래스 정리

| 문제 유형 | 예시 작업 | 적절한 모델 계열 | 대표 클래스 |
|---|---|---|---|
| 문장 전체에 라벨 붙이기 | 감성분석, 주제분류 | BERT 계열 | AutoModelForSequenceClassification |
| 각 토큰에 라벨 붙이기 | NER, POS 태깅 | BERT 계열 | AutoModelForTokenClassification |
| 문장 중간 빈칸 예측 | Fill-Mask | BERT 계열 | AutoModelForMaskedLM |
| 입력 문장을 다른 문장으로 변환 | 번역, 요약 | T5, BART, MarianMT 계열 | AutoModelForSeq2SeqLM |
| 앞 문맥을 보고 이어쓰기 | 문장 생성, 챗봇 | GPT 계열 | AutoModelForCausalLM |
| 벡터 표현 추출 | 문장 임베딩, 검색 | BERT 계열 또는 임베딩 특화 모델 | AutoModel |
